[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/metaflow-certified/notebooks/day-05-scaling-resources.ipynb#scrollTo=10a2b3c4)

---
# Day 5 · Scaling with @resources and @batch
**certified-journeys / metaflow-certified** · Metaflow for ML Engineers

> **Goal for today:** Annotate steps with `@resources`, understand how to target AWS Batch and Kubernetes with CLI flags, and learn how `@environment` passes env vars to remote tasks — all verifiable in local mode.


In [ ]:
%pip install -q metaflow


## Step 1 · The `@resources` Decorator

`@resources` declares the compute requirements for a step. Metaflow uses this declaration when scheduling the step on a remote executor (AWS Batch, Kubernetes). In **local mode** the decorator is parsed but has no effect — your laptop runs whatever it has.

| Parameter | Type | Meaning |
|-----------|------|---------|
| `cpu` | int | Number of CPU cores |
| `memory` | int | RAM in **MiB** (not GB) |
| `gpu` | int | Number of GPUs |
| `disk` | int | Ephemeral disk in MiB |

> **Rule of thumb:** Only annotate steps that truly need extra resources. Local steps stay fast while expensive compute runs remotely.


In [ ]:
%%writefile resources_flow.py
from metaflow import FlowSpec, step, resources

class ResourcesFlow(FlowSpec):
    """
    Shows @resources on a heavy training step while keeping
    lightweight steps un-annotated.
    """

    @step
    def start(self):
        # Lightweight preprocessing — no @resources needed
        print("start: loading dataset metadata")
        self.dataset_path = "s3://my-bucket/data/train.parquet"  # illustrative
        self.n_estimators_list = [50, 100, 200]
        self.next(self.train, foreach="n_estimators_list")

    @resources(cpu=4, memory=8192)  # 4 cores, 8 GiB — only requested on remote execution
    @step
    def train(self):
        # Simulate a memory-hungry training job
        import random
        random.seed(self.input)
        self.n_estimators = self.input
        # In production this would be: model = RandomForestClassifier(n_estimators=self.input).fit(X, y)
        self.val_accuracy = round(random.uniform(0.80, 0.97), 4)
        print(f"  Trained n_estimators={self.n_estimators} → val_acc={self.val_accuracy}")
        self.next(self.join)

    @step
    def join(self, inputs):
        # Another lightweight step — gather results
        self.results = [
            {"n_estimators": inp.n_estimators, "val_accuracy": inp.val_accuracy}
            for inp in inputs
        ]
        best = max(self.results, key=lambda r: r["val_accuracy"])
        self.best_n_estimators = best["n_estimators"]
        self.next(self.end)

    @step
    def end(self):
        print("Results:", self.results)
        print("Best n_estimators:", self.best_n_estimators)

if __name__ == "__main__":
    ResourcesFlow()


In [ ]:
# Run locally — @resources is declared but has no effect in local mode
!python resources_flow.py run


### What just happened?

- **`@resources(cpu=4, memory=8192)`** decorates only the `train` step — the annotation is stored in the flow's metadata and sent to the remote scheduler when you use `--with batch` or `--with kubernetes`.
- `memory=8192` is **MiB** (≈ 8 GiB). A common mistake is passing GB — double-check the unit.
- In local mode the flow runs fine because Metaflow ignores resource requests — you can always develop and test locally without cloud access.
- Decorator **order matters**: `@resources` must be placed *above* `@step` (outermost decorator applied last).


## Step 2 · Targeting AWS Batch with `--with batch`

Metaflow separates **what** (the flow code) from **where** (the executor). You never change the flow file to switch executors — you add a CLI flag.

```bash
# Run every step on AWS Batch:
python my_flow.py run --with batch

# Run only steps that have @batch on them:
python my_flow.py run
```

The `@batch` decorator can also be applied per-step to send just that step to AWS Batch while others run locally:

```python
from metaflow import FlowSpec, step, batch, resources

class MyFlow(FlowSpec):
    @batch            # always run this step on AWS Batch
    @resources(cpu=8, memory=16384)
    @step
    def heavy_step(self): ...
```

**Prerequisites for `--with batch`:**
- An AWS account with a configured Batch job queue and compute environment.
- `AWS_DEFAULT_REGION`, `METAFLOW_BATCH_JOB_QUEUE`, and `METAFLOW_ECS_S3_ACCESS_IAM_ROLE` set (or configured via `metaflow configure aws`).
- An S3 bucket for the datastore (`METAFLOW_DATASTORE_SYSROOT_S3`).


In [ ]:
%%writefile batch_demo_flow.py
"""
This file demonstrates the @batch decorator syntax.
Run locally for syntax validation — cloud execution requires AWS configuration.
"""
from metaflow import FlowSpec, step, resources
# `batch` is imported here to show usage — won't be applied at runtime without AWS config
try:
    from metaflow import batch
    HAS_BATCH = True
except ImportError:
    HAS_BATCH = False

class BatchDemoFlow(FlowSpec):
    """
    Illustrates per-step @batch targeting.
    Only the annotated step runs on Batch; other steps run locally.

    Production invocation:
        python batch_demo_flow.py run
        # or override all steps:
        python batch_demo_flow.py run --with batch
    """

    @step
    def start(self):
        print("start: runs locally")
        self.data = list(range(10))
        self.next(self.compute)

    # In production, uncomment @batch and @resources:
    # @batch
    # @resources(cpu=4, memory=8192)
    @step
    def compute(self):
        # This step would run on AWS Batch with 4 CPUs and 8 GiB RAM
        print(f"compute: processing {len(self.data)} items")
        self.squared = [x ** 2 for x in self.data]
        self.next(self.end)

    @step
    def end(self):
        print("end: runs locally")
        print("Squared:", self.squared)

if __name__ == "__main__":
    BatchDemoFlow()


In [ ]:
# Validate the flow locally (same code, different executor in production)
!python batch_demo_flow.py run


### What just happened?

- The flow is **identical** whether running locally or on Batch — only the CLI flag changes the executor.
- **`--with batch`** is a convenience override: it applies `@batch` to every step at runtime without modifying source code.
- Per-step `@batch` is more common for production: lightweight steps run locally (fast, free), heavy steps run on Batch (powerful, cost-effective).
- Metaflow packages your code and ships it to Batch automatically — no Docker build step required for standard Python flows.


## Step 3 · Targeting Kubernetes with `--with kubernetes`

Kubernetes support follows the same pattern as Batch — swap `batch` for `kubernetes`:

```bash
# Run all steps on Kubernetes:
python my_flow.py run --with kubernetes

# Per-step via decorator:
from metaflow import kubernetes

@kubernetes(cpu=4, memory=8192, namespace="ml-jobs")
@step
def heavy_step(self): ...
```

**Key differences — Batch vs. Kubernetes:**

| Feature | AWS Batch | Kubernetes |
|---------|-----------|------------|
| Scheduler | AWS-managed | Your cluster |
| Spot / preemptible | Yes (spot instances) | Yes (spot nodes) |
| Namespace support | No | Yes |
| GPU nodes | Yes | Yes |
| Config flag | `--with batch` | `--with kubernetes` |
| Profile setup | `metaflow configure aws` | `metaflow configure kubernetes` |

> **Tip:** Both executors honour `@resources` — you don't change the resource annotation when switching between Batch and Kubernetes.


In [ ]:
%%writefile kubernetes_demo_flow.py
"""
Demonstrates @kubernetes decorator syntax.
Runs locally for validation; cloud execution requires a configured K8s cluster.
"""
from metaflow import FlowSpec, step, resources

class KubernetesDemoFlow(FlowSpec):
    """
    Production invocation:
        python kubernetes_demo_flow.py run --with kubernetes

    Or with profile to pick a namespace:
        python kubernetes_demo_flow.py --with 'kubernetes:namespace=ml-prod' run
    """

    @step
    def start(self):
        print("Preparing job parameters")
        self.hyperparams = [{"lr": 0.001, "epochs": 10}, {"lr": 0.01, "epochs": 20}]
        self.next(self.train_gpu, foreach="hyperparams")

    # In production, uncomment these two decorators:
    # @kubernetes(cpu=8, memory=32768, gpu=1, namespace="ml-training")
    # @resources(gpu=1, memory=32768)
    @step
    def train_gpu(self):
        hp = self.input
        print(f"  Would train with lr={hp['lr']}, epochs={hp['epochs']} on GPU")
        # Production: run actual PyTorch/TF training here
        self.final_loss = round(0.5 / hp["lr"] * 0.001, 6)
        self.next(self.aggregate)

    @step
    def aggregate(self, inputs):
        self.run_results = [{"hp": inp.input, "loss": inp.final_loss} for inp in inputs]
        self.next(self.end)

    @step
    def end(self):
        for r in self.run_results:
            print(f"  hp={r['hp']}  loss={r['loss']}")

if __name__ == "__main__":
    KubernetesDemoFlow()


In [ ]:
!python kubernetes_demo_flow.py run


### What just happened?

- The `@kubernetes` decorator accepts all `@resources` parameters **plus** Kubernetes-specific ones like `namespace` and `image`.
- GPU allocation uses the same `gpu=1` parameter in both `@batch` and `@kubernetes` — Metaflow normalises the interface.
- Switching from Batch to Kubernetes is a one-line CLI change — the flow code is untouched.
- Running locally confirms the flow logic is correct before spending cloud credits.


## Step 4 · `@environment` — Passing Env Vars to Remote Steps

`@environment` injects environment variables into a step at execution time. This is the recommended way to pass configuration (API keys, feature flags, bucket names) to remote tasks without hardcoding them.

```python
from metaflow import environment

@environment(vars={"WANDB_API_KEY": "<secret>", "MODEL_BUCKET": "s3://my-bucket"})
@step
def train(self): ...
```

> **Security note:** Never hardcode secrets in `vars={}` in source code. Instead, read from `os.environ` at flow invocation time and interpolate:
> ```python
> import os
> @environment(vars={"WANDB_API_KEY": os.environ.get("WANDB_API_KEY", "")})
> ```


In [ ]:
%%writefile environment_flow.py
import os
from metaflow import FlowSpec, step, environment

# Read configuration from the launching environment
MODEL_BUCKET = os.environ.get("MODEL_BUCKET", "s3://default-bucket")
EXPERIMENT_TAG = os.environ.get("EXPERIMENT_TAG", "dev")

class EnvironmentFlow(FlowSpec):
    """
    Demonstrates passing env vars to remote steps via @environment.
    """

    @step
    def start(self):
        print("start: no special env vars needed here")
        self.next(self.train)

    @environment(vars={
        "MODEL_BUCKET": MODEL_BUCKET,      # resolved at launch time, not hardcoded
        "EXPERIMENT_TAG": EXPERIMENT_TAG,
        "OMP_NUM_THREADS": "4",            # control OpenMP threading in the remote container
    })
    @resources(cpu=4, memory=8192)
    @step
    def train(self):
        # Inside the remote task these env vars are set correctly
        bucket = os.environ.get("MODEL_BUCKET", "not-set")
        tag = os.environ.get("EXPERIMENT_TAG", "not-set")
        omp = os.environ.get("OMP_NUM_THREADS", "not-set")
        print(f"  MODEL_BUCKET    = {bucket}")
        print(f"  EXPERIMENT_TAG  = {tag}")
        print(f"  OMP_NUM_THREADS = {omp}")
        self.model_path = f"{bucket}/models/{tag}/model.pkl"
        self.next(self.end)

    @step
    def end(self):
        print("Model would be saved to:", self.model_path)

if __name__ == "__main__":
    EnvironmentFlow()


In [ ]:
# Run with custom env vars
!MODEL_BUCKET="s3://prod-models" EXPERIMENT_TAG="v1.0" python environment_flow.py run


### What just happened?

- **`@environment(vars={...})`** serialises the dict and ships it with the task to the remote executor — the variables are set in the container before the step function runs.
- Reading `os.environ` at the module level (outside the class) captures the *launcher's* environment at submission time — a safe pattern for secrets.
- `OMP_NUM_THREADS` is a good example of a non-secret env var — it tunes thread count in numerical libraries (NumPy, SciPy) to match the CPUs requested via `@resources`.
- Env vars set by `@environment` do **not** persist across steps — each step's environment is isolated.


## Step 5 · Putting It Together — a Real Scaling Pattern

A typical production ML flow pattern combines all three decorators: `@resources` to size the compute, `@environment` for config, and a foreach for parallelism — then inspects the run via the Client API.


In [ ]:
%%writefile scaling_pattern_flow.py
import os
from metaflow import FlowSpec, step, resources, environment

DATA_BUCKET = os.environ.get("DATA_BUCKET", "s3://example-bucket")

class ScalingPatternFlow(FlowSpec):
    """
    Production-ready scaling pattern:
    - Lightweight start/end steps (no @resources)
    - Heavy foreach training steps with @resources + @environment
    - Final aggregation step

    To scale to cloud:
        python scaling_pattern_flow.py run --with batch
        python scaling_pattern_flow.py run --with kubernetes
    """

    @step
    def start(self):
        self.configs = [
            {"model": "xgb", "depth": 6},
            {"model": "lgbm", "depth": 8},
            {"model": "catboost", "depth": 10},
        ]
        self.next(self.train_and_eval, foreach="configs")

    @environment(vars={"DATA_BUCKET": DATA_BUCKET, "OMP_NUM_THREADS": "4"})
    @resources(cpu=4, memory=8192)
    @step
    def train_and_eval(self):
        import random, os
        cfg = self.input
        random.seed(hash(cfg["model"]) + cfg["depth"])
        self.model_name = cfg["model"]
        self.depth = cfg["depth"]
        self.val_auc = round(random.uniform(0.82, 0.96), 4)
        self.data_bucket_used = os.environ.get("DATA_BUCKET", "not-set")
        print(f"  {self.model_name} depth={self.depth}: AUC={self.val_auc}")
        print(f"  Read data from: {self.data_bucket_used}")
        self.next(self.aggregate)

    @step
    def aggregate(self, inputs):
        self.leaderboard = sorted(
            [{"model": inp.model_name, "depth": inp.depth, "auc": inp.val_auc}
             for inp in inputs],
            key=lambda r: -r["auc"]
        )
        self.next(self.end)

    @step
    def end(self):
        print("\nFinal leaderboard:")
        for rank, r in enumerate(self.leaderboard, 1):
            print(f"  #{rank} {r['model']} (depth={r['depth']}): AUC={r['auc']}")

if __name__ == "__main__":
    ScalingPatternFlow()


In [ ]:
!python scaling_pattern_flow.py run


In [ ]:
# Inspect the run using the Client API
from metaflow import Flow

run = Flow("ScalingPatternFlow").latest_run
print(f"Run: {run.id}")

# Show decorator metadata for the training step
train_step = run["train_and_eval"]
print(f"\nTasks in 'train_and_eval':")
for task in train_step:
    print(f"  task {task.id}: model={task.data.model_name}, auc={task.data.val_auc}")

# Show the final leaderboard artifact
agg_task = next(iter(run["aggregate"]))
print("\nLeaderboard artifact:")
for entry in agg_task.data.leaderboard:
    print(f"  {entry}")


### What just happened?

- **All three decorators** (`@environment`, `@resources`, `@step`) stack cleanly — outermost to innermost, with `@step` always at the bottom.
- The **Client API** is independent of where the flow ran — the same `Flow('...').latest_run` call works for local, Batch, and Kubernetes runs.
- **Per-task artifacts** (`model_name`, `val_auc`) are stored individually for each foreach task — you can always replay or compare specific tasks.
- The complete scaling strategy: develop locally → validate logic → run with `--with batch` for production scale — zero code changes.


In [ ]:
# Challenge: Add a @resources(cpu=2, memory=4096) annotation to the aggregate step,
# then add an @environment that passes a RESULT_BUCKET env var.
# In the aggregate step, print the value of RESULT_BUCKET from os.environ.
#
# Scaffold:
# import os
# RESULT_BUCKET = os.environ.get("RESULT_BUCKET", "s3://results")
#
# @environment(vars={"RESULT_BUCKET": ???})
# @resources(cpu=???, memory=???)
# @step
# def aggregate(self, inputs):
#     bucket = os.environ.get("RESULT_BUCKET", "not-set")
#     ...
pass


---
## Day 5 key concepts recap

| Concept | What to remember |
|---|---|
| `@resources(cpu, memory)` | Declares compute needs — MiB for memory, ignored in local mode |
| `--with batch` | Sends every step to AWS Batch without changing flow code |
| `@batch` per step | Targets only that step to Batch; others run locally |
| `--with kubernetes` | Same pattern as Batch but for a K8s cluster |
| `@kubernetes` per step | Supports extra params: `namespace`, `image` |
| `@environment(vars={})` | Injects env vars into the remote container at execution time |
| Decorator order | `@environment` → `@resources` → `@step` (outermost to innermost) |

> **Tip:** Use `@resources` only on steps that need it — local steps stay fast while expensive compute runs remotely.

---
## What's next
**Day 6** → Pin Python dependencies per step using `@conda` and `@pypi`, so each step carries its exact environment and reproducibility is guaranteed.

Mark Day 5 complete in your [tracker](../index.html).
